In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Output, Layout
from IPython.display import display, clear_output

# ============================================================
# REAL QUANTIZATION NOISE VS PQN MODEL
# ============================================================

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>

.pqn-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 18px !important;
    align-items: center !important;
}

.pqn-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}

.pqn-radio > label {
    display: none !important;
}

</style>
"""))

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:540px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#9a4f21;
    margin-bottom:8px;
">
Real Quantization Noise and the PQN Model
</div>

<div style="margin-bottom:5px;">
The actual quantization error is ν[n] = Q(x[n]) − x[n] and is therefore a deterministic function of the input.
</div>

<div style="margin-bottom:5px;">
The PQN model replaces this error by an independent uniform random variable n[n] distributed over [−Δ/2, Δ/2].
</div>

<div style="margin-bottom:5px;">
Ideal PQN has E{n} = 0 and Var{n} = Δ²/12, but the actual quantization error does not necessarily possess these properties.
</div>

<div>
<b>This notebook:</b> compares the real quantization error with the ideal PQN approximation for several types of input.
</div>

</div>
""")

# ============================================================
# INPUT MODEL
# ============================================================

input_selector = RadioButtons(
    options=['Gaussian', 'Uniform', 'Sinusoidal'],
    value='Gaussian',
    description='',
    layout=Layout(width='300px')
)

input_selector.add_class('pqn-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='135px',
    min_width='135px'
)

delta_slider = FloatSlider(
    min=0.20,
    max=2.00,
    step=0.10,
    value=1.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

scale_slider = FloatSlider(
    min=0.25,
    max=3.00,
    step=0.25,
    value=1.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

frequency_slider = FloatSlider(
    min=0.01,
    max=0.20,
    step=0.01,
    value=0.05,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

N_slider = IntSlider(
    min=500,
    max=5000,
    step=500,
    value=2500,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# VALUE LABELS
# ============================================================

value_layout = Layout(
    width='55px',
    min_width='55px',
    margin='0px 0px 0px 4px'
)

delta_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>',
    layout=value_layout
)

scale_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>',
    layout=value_layout
)

frequency_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.05</div>',
    layout=value_layout
)

N_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">2500</div>',
    layout=value_layout
)

# ============================================================
# LABELS
# ============================================================

label_layout = Layout(
    width='150px',
    min_width='150px'
)

input_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Input:</div>',
    layout=label_layout
)

delta_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Quantization step Δ:</div>',
    layout=label_layout
)

scale_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Input scale:</div>',
    layout=label_layout
)

frequency_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Normalized freq. f₀:</div>',
    layout=label_layout
)

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Samples N:</div>',
    layout=label_layout
)

# ============================================================
# CONTROL ROWS
# ============================================================

row_layout = Layout(
    width='465px',
    min_width='465px',
    height='36px',
    min_height='36px',
    align_items='center',
    overflow='visible'
)

input_row = HBox(
    [
        input_label,
        input_selector
    ],
    layout=row_layout
)

delta_row = HBox(
    [
        delta_label,
        delta_slider,
        delta_value
    ],
    layout=row_layout
)

scale_row = HBox(
    [
        scale_label,
        scale_slider,
        scale_value
    ],
    layout=row_layout
)

frequency_row = HBox(
    [
        frequency_label,
        frequency_slider,
        frequency_value
    ],
    layout=row_layout
)

N_row = HBox(
    [
        N_label,
        N_slider,
        N_value
    ],
    layout=row_layout
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#9a4f21;
            margin-bottom:8px;
        ">
        Quantization Parameters
        </div>
        """),

        input_row,
        delta_row,
        scale_row,
        frequency_row,
        N_row
    ],
    layout=Layout(
        width='500px',
        min_width='500px',
        padding='10px 12px 12px 12px',
        border='1px solid #dfc7b7',
        overflow='visible'
    )
)

# ============================================================
# TOP TWO-COLUMN LAYOUT
#
# LEFT  : documentation
# RIGHT : quantization controls
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1080px',
        align_items='flex-start',
        justify_content='space-between',
        gap='16px',
        margin='0px 0px 10px 0px',
        overflow='visible'
    )
)

# ============================================================
# OUTPUT AREAS
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

result_html = HTML()

# ============================================================
# MID-TREAD QUANTIZER
# ============================================================

def quantize(x, delta):

    return delta * np.floor(
        x / delta + 0.5
    )

# ============================================================
# NORMALIZED AUTOCORRELATION
# ============================================================

def normalized_acf(x, max_lag):

    x = x - np.mean(x)

    N = len(x)

    variance = np.mean(
        x**2
    )

    acf = np.zeros(
        max_lag + 1
    )

    if variance == 0:
        return acf

    for lag in range(max_lag + 1):

        acf[lag] = np.mean(
            x[:N - lag] * x[lag:]
        ) / variance

    return acf

# ============================================================
# MAIN PLOT FUNCTION
# ============================================================

def plot_pqn_comparison(input_type, delta, scale, frequency, N):

    # --------------------------------------------------------
    # FIXED RANDOM GENERATORS
    # --------------------------------------------------------

    rng_input = np.random.default_rng(731)

    rng_pqn = np.random.default_rng(117)

    # --------------------------------------------------------
    # INPUT SIGNAL / RANDOM SEQUENCE
    # --------------------------------------------------------

    n = np.arange(
        N
    )

    if input_type == 'Gaussian':

        x = scale * rng_input.standard_normal(
            N
        )

        input_description = f'Gaussian input: σ = {scale:.2f}'

    elif input_type == 'Uniform':

        half_width = np.sqrt(3.0) * scale

        x = rng_input.uniform(
            -half_width,
            half_width,
            N
        )

        input_description = f'Uniform input: σ ≈ {scale:.2f}'

    else:

        x = scale * np.sin(
            2.0 * np.pi * frequency * n
        )

        input_description = f'Sinusoidal input: A = {scale:.2f}, f₀ = {frequency:.3f}'

    # --------------------------------------------------------
    # REAL QUANTIZATION
    # --------------------------------------------------------

    y = quantize(
        x,
        delta
    )

    nu = y - x

    # --------------------------------------------------------
    # IDEAL PQN
    # --------------------------------------------------------

    pqn = rng_pqn.uniform(
        -delta / 2.0,
        delta / 2.0,
        N
    )

    # --------------------------------------------------------
    # STATISTICS
    # --------------------------------------------------------

    mean_real = np.mean(
        nu
    )

    variance_real = np.var(
        nu
    )

    mean_pqn = np.mean(
        pqn
    )

    variance_pqn = np.var(
        pqn
    )

    theoretical_variance = delta**2 / 12.0

    # --------------------------------------------------------
    # INPUT / ERROR CORRELATION
    # --------------------------------------------------------

    if np.std(x) > 0 and np.std(nu) > 0:

        rho_xnu = np.corrcoef(
            x,
            nu
        )[0, 1]

    else:

        rho_xnu = 0.0

    # --------------------------------------------------------
    # AUTOCORRELATIONS
    # --------------------------------------------------------

    max_lag = 30

    lags = np.arange(
        max_lag + 1
    )

    acf_real = normalized_acf(
        nu,
        max_lag
    )

    acf_pqn = normalized_acf(
        pqn,
        max_lag
    )

    # --------------------------------------------------------
    # DISPLAY SAMPLES
    # --------------------------------------------------------

    display_N = min(
        220,
        N
    )

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.6, 7.3)
    )

    gs = fig.add_gridspec(
        2,
        2,
        hspace=0.46,
        wspace=0.30
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, 0]
    )

    ax4 = fig.add_subplot(
        gs[1, 1]
    )

    # ========================================================
    # GRAPH 1:
    # INPUT AND QUANTIZED OUTPUT
    # ========================================================

    ax1.plot(
        n[:display_N],
        x[:display_N],
        linewidth=1.2,
        label='Input x[n]'
    )

    ax1.step(
        n[:display_N],
        y[:display_N],
        where='mid',
        linewidth=1.2,
        label='Quantized y[n]'
    )

    y_limit = max(
        4.0,
        3.5 * scale
    )

    ax1.set_xlim(
        0,
        display_N - 1
    )

    ax1.set_ylim(
        -y_limit,
        y_limit
    )

    ax1.set_xlabel(
        'Sample index n',
        fontsize=10
    )

    ax1.set_ylabel(
        'Amplitude',
        fontsize=10
    )

    ax1.set_title(
        'Input and Quantized Output',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax1.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # GRAPH 2:
    # ACTUAL QUANTIZATION ERROR
    # ========================================================

    ax2.plot(
        n[:display_N],
        nu[:display_N],
        linewidth=1.1
    )

    ax2.axhline(
        delta / 2.0,
        linestyle='--',
        linewidth=1.0,
        label='+Δ/2'
    )

    ax2.axhline(
        -delta / 2.0,
        linestyle='--',
        linewidth=1.0,
        label='−Δ/2'
    )

    ax2.axhline(
        0,
        linewidth=0.8
    )

    ax2.set_xlim(
        0,
        display_N - 1
    )

    ax2.set_ylim(
        -1.1,
        1.1
    )

    ax2.set_xlabel(
        'Sample index n',
        fontsize=10
    )

    ax2.set_ylabel(
        'ν[n]',
        fontsize=10
    )

    ax2.set_title(
        'Actual Quantization Error',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax2.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # GRAPH 3:
    # ERROR PDF / HISTOGRAM
    # ========================================================

    bins = np.linspace(
        -delta / 2.0,
        delta / 2.0,
        31
    )

    ax3.hist(
        nu,
        bins=bins,
        density=True,
        alpha=0.60,
        label='Actual error'
    )

    ideal_x = np.array(
        [
            -delta / 2.0,
            -delta / 2.0,
            delta / 2.0,
            delta / 2.0
        ]
    )

    ideal_y = np.array(
        [
            0.0,
            1.0 / delta,
            1.0 / delta,
            0.0
        ]
    )

    ax3.plot(
        ideal_x,
        ideal_y,
        linewidth=2.0,
        label='Ideal PQN PDF'
    )

    ax3.set_xlim(
        -1.05,
        1.05
    )

    ax3.set_ylim(
        0,
        5.5
    )

    ax3.set_xlabel(
        'Error amplitude',
        fontsize=10
    )

    ax3.set_ylabel(
        'Probability density',
        fontsize=10
    )

    ax3.set_title(
        'Actual Error PDF vs Ideal PQN',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax3.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # GRAPH 4:
    # ERROR AUTOCORRELATION
    # ========================================================

    ax4.plot(
        lags,
        acf_real,
        marker='o',
        markersize=3,
        linewidth=1.5,
        label='Actual error'
    )

    ax4.plot(
        lags,
        acf_pqn,
        marker='s',
        markersize=3,
        linewidth=1.3,
        label='Ideal PQN realization'
    )

    ax4.axhline(
        0,
        linewidth=0.8
    )

    ax4.set_xlim(
        0,
        max_lag
    )

    ax4.set_ylim(
        -1.05,
        1.05
    )

    ax4.set_xlabel(
        'Lag k',
        fontsize=10
    )

    ax4.set_ylabel(
        'Normalized autocorrelation',
        fontsize=10
    )

    ax4.set_title(
        'Error Correlation Structure',
        fontsize=12,
        pad=8
    )

    ax4.tick_params(
        axis='both',
        labelsize=9
    )

    ax4.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax4.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.09
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.50;
        width:980px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>{input_description}</b>

    <br>

    <b>Actual error:</b>
    E{{ν}} = {mean_real:.5f}
    &nbsp;&nbsp;&nbsp;
    Var{{ν}} = {variance_real:.5f}

    <br>

    <b>Ideal PQN:</b>
    E{{n}} = 0
    &nbsp;&nbsp;&nbsp;
    Var{{n}} = Δ²/12 = {theoretical_variance:.5f}

    <br>

    <b>Generated PQN realization:</b>
    mean = {mean_pqn:.5f}
    &nbsp;&nbsp;&nbsp;
    variance = {variance_pqn:.5f}

    <br>

    <b>Input / actual-error correlation coefficient:</b>
    ρ(x,ν) = {rho_xnu:.5f}

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    # --------------------------------------------------------
    # FREQUENCY IS USED ONLY FOR SINUSOIDAL INPUT
    # --------------------------------------------------------

    frequency_slider.disabled = (
        input_selector.value != 'Sinusoidal'
    )

    # --------------------------------------------------------
    # CURRENT VALUES
    # --------------------------------------------------------

    delta_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{delta_slider.value:.2f}</div>'

    scale_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{scale_slider.value:.2f}</div>'

    frequency_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{frequency_slider.value:.2f}</div>'

    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

    # --------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------

    with graph_output:

        clear_output(
            wait=True
        )

        plot_pqn_comparison(
            input_selector.value,
            delta_slider.value,
            scale_slider.value,
            frequency_slider.value,
            N_slider.value
        )

# ============================================================
# CONNECT CONTROLS
# ============================================================

input_selector.observe(
    update_notebook,
    names='value'
)

delta_slider.observe(
    update_notebook,
    names='value'
)

scale_slider.observe(
    update_notebook,
    names='value'
)

frequency_slider.observe(
    update_notebook,
    names='value'
)

N_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #e1cabc;
    background:#fff9f5;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#9a4f21;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The actual error ν = Q(x) − x always lies within approximately [−Δ/2, Δ/2], but this alone does not imply that it is uniformly distributed.
</div>

<div style="margin-bottom:4px;">
The PQN approximation assumes a uniform, zero-mean, input-independent noise with variance Δ²/12.
</div>

<div>
Agreement of the actual-error histogram, moments and correlation structure with the PQN model determines how accurate this stochastic approximation is for the selected input.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
#
# THEORY + CONTROLS SIDE BY SIDE
# GRAPHS BELOW
# ============================================================

main_layout = VBox(
    [
        top_layout,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='visible'
    )
)

display(main_layout)